In [ ]:
%pip install semantic-link-labs --q


In [ ]:
# Imports
from pyspark.sql.functions import *
from pyspark.sql import Row
from sempy_labs import admin
import sempy.fabric as fabric


In [ ]:
# Get the PPU (PP3) Capacity
capacity_id = (
    spark
        .createDataFrame(admin.list_capacities())
        .where(col("Sku") == lit("PP3"))
        .select("Capacity Id")
        .collect()[0][0]
)


In [ ]:
# Get the list of Workspaces assigned to PPU
df_workspaces = (
    spark
        .createDataFrame(admin.list_workspaces())
        .where(col("Capacity Id") == lit(capacity_id))
        .withColumnRenamed("Id", "Workspace Id")
        .withColumnRenamed("Name", "Workspace Name")
        .select("Workspace Id",
                "Workspace Name"
        )
)


In [ ]:
# Get the list of Semantic Models (Datasets) in the PPU Workspaces
df_list_datasets = (
    spark
        .createDataFrame(admin.list_datasets())
        .join(df_workspaces, on="Workspace Id", how="left")
        .where(~col("Content Provider Type").isin(["Unknown" # E.g.: Fabric Capacity Metrics app.
                                                  ])
        )
        .orderBy("Workspace Name", "Dataset Name")
)

display (
    df_list_datasets
)


In [ ]:
# Iterate over all Semantic Models (Datasets) in the PPU Workspaces and calculate its size
rows = []

for r in df_list_datasets.collect():
    workspace_name = r["Workspace Name"]
    workspace_id   = r["Workspace Id"]
    dataset_name   = r["Dataset Name"]
    dataset_id     = r["Dataset Id"]

    print(f"▶️ Starting analyzing Workspace Name = '{workspace_name}', Dataset Name = '{dataset_name}', Workspace Id = '{workspace_id}', Dataset Id = '{dataset_id}'")

    try:
        # --- HARD SANDBOX: catch ANY exception, including XMLA session failures ---
        try:
            df_analysis = fabric.model_memory_analyzer(
                dataset=dataset_id,
                workspace=workspace_id,
                return_dataframe=True
            )
        except Exception as inner_exc:
            # Force the XMLA/DirectLake error into a normal Python exception
            raise RuntimeError(f"Analyzer failed: {inner_exc}")

        # Extract Total Size
        total_size = df_analysis["Model Summary"]["Total Size"].iloc[0]

        rows.append(
            Row(
                workspace_name = workspace_name,
                workspace_id   = workspace_id,
                dataset_name   = dataset_name,
                dataset_id     = dataset_id,
                total_size     = int(total_size)
            )
        )

    except Exception as exc:
        print(f"❌ Error analyzing Workspace Name = '{workspace_name}', Dataset Name = '{dataset_name}', Workspace Id = '{workspace_id}', Dataset Id = '{dataset_id}': '{exc}'")

        rows.append(
            Row(
                workspace_name = workspace_name,
                workspace_id   = workspace_id,
                dataset_name   = dataset_name,
                dataset_id     = dataset_id,
                total_size     = None
            )
        )
        # continue is implicit



In [ ]:
# List the size of all Semantic Models (Datasets) in the PPU Workspaces
df_dataset_sizes = (
    spark
        .createDataFrame(rows)
        .withColumnRenamed("workspace_name", "Workspace Name")
        .withColumnRenamed("workspace_id", "Workspace Id")
        .withColumnRenamed("dataset_name", "Dataset Name")
        .withColumnRenamed("dataset_id", "Dataset Id")
        .withColumnRenamed("total_size", "Total Size (bytes)")
)

display (
    df_dataset_sizes
)


In [ ]:
# Aggregate the size of all Semantic Models (Datasets) per Workspace and total at Tenant level
display (
    df_dataset_sizes
        .groupBy("Workspace Name")
        .agg(
            round(sum(col("Total Size (bytes)")) / 1024.0 / 1024.0, 2).alias("Total Size (MB)"),
            round(sum(col("Total Size (bytes)")) / 1024.0 / 1024.0 / 1024.0, 2).alias("Total Size (GB)")
        )
        .union(
                df_dataset_sizes
                    .agg(
                        round(sum(col("Total Size (bytes)")) / 1024.0 / 1024.0, 2).alias("Total Size (MB)"),
                        round(sum(col("Total Size (bytes)")) / 1024.0 / 1024.0 / 1024.0, 2).alias("Total Size (GB)")
                    )
                    .withColumn("Workspace Name", lit("PPU at Tenant"))
                    .select("Workspace Name",
                            "Total Size (MB)",
                            "Total Size (GB)"
                    )
        )
)
